# The Mathematical Building Blocks of Neural Networks

### A first look at a neural network

This section introduces a basic neural network using the Keras library to classify handwritten digits from the MNIST dataset. We'll load the data, define a simple model, train it, and then evaluate its performance.

In [28]:
from keras.datasets import mnist
import numpy as np

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

### Loading the MNIST Dataset

We start by loading the MNIST dataset, which consists of 60,000 training images and 10,000 test images of handwritten digits (0-9). Each image is a 28x28 pixel grayscale image.

In [29]:
train_images.shape

(60000, 28, 28)

In [30]:
len(train_labels)

60000

In [31]:
train_labels

array([5, 0, 4, ..., 5, 6, 8], dtype=uint8)

### Data Exploration

Let's inspect the shape and content of our loaded dataset to understand its structure.

In [32]:
test_images.shape

(10000, 28, 28)

In [33]:
len(test_labels)

10000

In [34]:
test_labels

array([7, 2, 1, ..., 4, 5, 6], dtype=uint8)

### Defining the Neural Network Model

We define a simple sequential neural network using Keras. It consists of two `Dense` layers:
- The first `Dense` layer has 512 units and uses the 'relu' activation function.
- The second `Dense` layer has 10 units (one for each digit class) and uses the 'softmax' activation function to output probabilities for each class.

In [35]:
import keras
from keras import layers

model = keras.Sequential(
    [
        layers.Dense(512, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)

### Compiling the Model

Before training, we need to compile the model. We specify:
- An `optimizer`: 'adam' is a popular choice.
- A `loss function`: 'sparse_categorical_crossentropy' is suitable for multi-class classification when labels are integers.
- `metrics`: We'll track 'accuracy' during training.

In [36]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

### Data Preprocessing

To prepare the images for the neural network, we perform the following steps:
1.  **Reshape**: Convert the 28x28 pixel images into a 1D vector of 784 pixels. This is because our `Dense` layer expects a 1D input.
2.  **Normalize**: Convert the pixel values from integers (0-255) to floating-point numbers between 0 and 1. This helps the network learn more effectively.

In [37]:
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

### Training the Model

Now, we train the model using the `fit` method. We provide the preprocessed `train_images` and `train_labels`, specify the number of `epochs` (how many times to iterate over the entire dataset), and the `batch_size` (number of samples per gradient update).

In [38]:
model.fit(train_images, train_labels, epochs=5, batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9242 - loss: 0.2665
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9679 - loss: 0.1090
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9794 - loss: 0.0711
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9856 - loss: 0.0495
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9893 - loss: 0.0375


### Making Predictions and Evaluating the Model

After training, we can use the model to make predictions on new data (e.g., the first few test images) and evaluate its overall performance on the unseen `test_images`.

In [39]:
test_digits = test_images[0:10]
predictions = model.predict(test_digits)
predictions[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step


array([7.2659299e-07, 5.8519536e-08, 3.1489228e-05, 2.6192972e-03,
       1.1214282e-09, 1.3039108e-06, 5.3802782e-11, 9.9733210e-01,
       6.6039829e-06, 8.4827470e-06], dtype=float32)

In [40]:
predictions[0].argmax()

np.int64(7)

In [41]:
predictions[0][7]

np.float32(0.9973321)

In [42]:
test_labels[0]

np.uint8(7)

### Model Evaluation on Test Data

Finally, we evaluate the model's performance on the entire test set to get an unbiased estimate of its generalization capability. We print the `test_acc` (test accuracy).

In [43]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"test_acc: {test_acc}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9801 - loss: 0.0642
test_acc: 0.9800999760627747


#### Reimplementing our first example from scratch

### Reimplementing Our First Example from Scratch

This section demonstrates how to build the core components of a neural network manually, mirroring the functionality of Keras. This provides a deeper understanding of how these networks work under the hood.

##### A simple Dense class

#### A Simple `Dense` Layer Class

We define a `NaiveDense` class that mimics Keras's `Dense` layer. It handles:
- **Initialization**: Setting up weights (`W`) and biases (`b`) with random uniform or zero initializers.
- **Forward Pass (`__call__`)**: Performing matrix multiplication (`inputs * W + b`) and applying an activation function if specified.
- **Weights Property**: Providing access to the layer's trainable weights.

In [44]:
import keras
from keras import ops

class NaiveDense:
    def __init__(self, input_size, output_size, activation=None):
        self.activation = activation
        self.W = keras.Variable(
            shape=(input_size, output_size), initializer="uniform"
        )
        self.b = keras.Variable(shape=(output_size,), initializer="zeros")

    def __call__(self, inputs):
        x = ops.matmul(inputs, self.W)
        x = x + self.b
        if self.activation is not None:
            x = self.activation(x)
        return x

    @property
    def weights(self):
        return [self.W, self.b]

##### A simple Sequential class

#### A Simple `Sequential` Model Class

Similar to Keras's `Sequential` model, our `NaiveSequential` class chains together multiple `NaiveDense` layers. It:
- **Initialization**: Takes a list of layers.
- **Forward Pass (`__call__`)**: Passes the input through each layer sequentially.
- **Weights Property**: Aggregates all trainable weights from its constituent layers.

In [45]:
class NaiveSequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x

    @property
    def weights(self):
        weights = []
        for layer in self.layers:
            weights += layer.weights
        return weights

In [46]:
model = NaiveSequential(
    [
        NaiveDense(input_size=28 * 28, output_size=512, activation=ops.relu),
        NaiveDense(input_size=512, output_size=10, activation=ops.softmax),
    ]
)
assert len(model.weights) == 4

#### A Batch Generator

To train efficiently, we process data in mini-batches. The `BatchGenerator` class helps us iterate through the dataset, yielding batches of images and corresponding labels. This is crucial for stochastic gradient descent.

##### A batch generator

In [47]:
import math

class BatchGenerator:
    def __init__(self, images, labels, batch_size=128):
        assert len(images) == len(labels)
        self.index = 0
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        self.num_batches = math.ceil(len(images) / batch_size)

    def next(self):
        images = self.images[self.index : self.index + self.batch_size]
        labels = self.labels[self.index : self.index + self.batch_size]
        self.index += self.batch_size
        return images, labels

### Running One Training Step

This section delves into the mechanics of a single training step, involving weight updates and gradient computations.

#### Running one training step

##### The weight update step

#### The Weight Update Step

The `update_weights` function is responsible for adjusting the model's weights based on the computed gradients and a `learning_rate`. Initially, it's shown as a manual update, then replaced with Keras's `optimizers.SGD` for a more robust approach.

In [48]:
learning_rate = 1e-3

def update_weights(gradients, weights):
    for g, w in zip(gradients, weights):
        w.assign(w - g * learning_rate)

#### Gradient Computation

We use `tf.GradientTape` to automatically compute gradients of the loss with respect to the model's trainable weights. This is a powerful feature that allows us to implement backpropagation efficiently.

The `one_training_step` function encapsulates the forward pass, loss calculation, gradient computation, and weight update for a single batch.

In [49]:
from keras import optimizers

optimizer = optimizers.SGD(learning_rate=1e-3)

def update_weights(gradients, weights):
    optimizer.apply_gradients(zip(gradients, weights))

##### Gradient computation

In [50]:
import tensorflow as tf

x = tf.zeros(shape=())
with tf.GradientTape() as tape:
    y = 2 * x + 3
grad_of_y_wrt_x = tape.gradient(y, x)

In [51]:
def one_training_step(model, images_batch, labels_batch):
    with tf.GradientTape() as tape:
        predictions = model(images_batch)
        loss = ops.sparse_categorical_crossentropy(labels_batch, predictions)
        average_loss = ops.mean(loss)
    gradients = tape.gradient(average_loss, model.weights)
    update_weights(gradients, model.weights)
    return average_loss

#### The full training loop

### The Full Training Loop

Putting all the pieces together, the `fit` function orchestrates the entire training process. It iterates through epochs, creates batches of data using `BatchGenerator`, and calls `one_training_step` for each batch.

In [52]:
def fit(model, images, labels, epochs, batch_size=128):
    for epoch_counter in range(epochs):
        print(f"Epoch {epoch_counter}")
        batch_generator = BatchGenerator(images, labels)
        for batch_counter in range(batch_generator.num_batches):
            images_batch, labels_batch = batch_generator.next()
            loss = one_training_step(model, images_batch, labels_batch)
            if batch_counter % 100 == 0:
                print(f"loss at batch {batch_counter}: {loss:.2f}")

### Training the Naive Model

Here we run our custom-built training loop with the `NaiveDense` and `NaiveSequential` classes on the MNIST dataset. We observe the loss decreasing over epochs, indicating that the model is learning.

In [53]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

fit(model, train_images, train_labels, epochs=10, batch_size=128)

Epoch 0
loss at batch 0: 2.31
loss at batch 100: 2.27
loss at batch 200: 2.22
loss at batch 300: 2.20
loss at batch 400: 2.16
Epoch 1
loss at batch 0: 2.12
loss at batch 100: 2.11
loss at batch 200: 2.04
loss at batch 300: 2.02
loss at batch 400: 1.98
Epoch 2
loss at batch 0: 1.93
loss at batch 100: 1.95
loss at batch 200: 1.86
loss at batch 300: 1.84
loss at batch 400: 1.80
Epoch 3
loss at batch 0: 1.73
loss at batch 100: 1.77
loss at batch 200: 1.66
loss at batch 300: 1.65
loss at batch 400: 1.62
Epoch 4
loss at batch 0: 1.54
loss at batch 100: 1.59
loss at batch 200: 1.46
loss at batch 300: 1.46
loss at batch 400: 1.45
Epoch 5
loss at batch 0: 1.35
loss at batch 100: 1.42
loss at batch 200: 1.28
loss at batch 300: 1.29
loss at batch 400: 1.30
Epoch 6
loss at batch 0: 1.20
loss at batch 100: 1.26
loss at batch 200: 1.12
loss at batch 300: 1.14
loss at batch 400: 1.17
Epoch 7
loss at batch 0: 1.06
loss at batch 100: 1.13
loss at batch 200: 0.99
loss at batch 300: 1.02
loss at batch 40

#### Evaluating the model

### Evaluating the Naive Model

Finally, we evaluate the accuracy of our manually trained model on the test dataset. We compare the predicted labels (obtained by taking the argmax of the model's output probabilities) with the true `test_labels` to calculate the accuracy.

In [54]:
predictions = model(test_images)
predicted_labels = ops.argmax(predictions, axis=1)
matches = predicted_labels == test_labels
f"accuracy: {ops.mean(matches):.2f}"

'accuracy: 0.83'